<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 5: Pandas, agrupaciones y combinación</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

## Conceptos

1. Resumir una tabla por grupo con `groupby` y pedir varias métricas a la vez con `agg`.
2. Elegir entre promedio, mediana, suma y recuento según lo que se quiera afirmar, y no por costumbre.
3. Combinar tablas con `merge`, controlando cuántas filas entran y cuántas salen.
4. Detectar lo que un cruce mal hecho hace desaparecer en silencio, y lo que duplica sin avisar.

<h2 style='font-weight:bold; color:#003366;'>Dinámica de clase</h2>

Segunda parte de la misma investigación. Las mesas son las mismas y los casos son los mismos pero cada caso viene con tablas adicionales. 

La puesta en común es contar los hallazgos de la parte 1 y **defender una recomendación** frente al otro grupo.

<h2 style='font-weight:bold; color:#003366;'>1. Teórico: agrupar y cruzar</h2>

Usa los mismos datos que el teórico anterior de `ventas_ecommerce.csv`, para ver las tres funcionalidades nuevas antes de aplicarlas: agrupar, pedir varias métricas juntas y cruzar con otra tabla.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def ruta_datos(nombre_archivo, subcarpeta=""):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / subcarpeta / nombre_archivo,
        Path("Datos") / subcarpeta / nombre_archivo,
        Path("/content/Datos") / subcarpeta / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


ventas = pd.read_csv(ruta_datos("ventas_ecommerce.csv"), parse_dates=["fecha"])
ventas["facturacion"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento"])
print("Filas:", len(ventas))

### Agrupar

`groupby` parte la tabla en grupos según los valores de una o más columnas y permite aplicar funciones sobre cada grupo.

Cuando usamos **`agg`**, las métricas **resumen cada grupo**, por lo que normalmente obtenemos una fila por grupo. Es útil cuando queremos construir una tabla resumen.

Cuando usamos **`transform`**, la métrica se calcula a nivel de grupo pero **se conserva la cantidad de filas original**. Es útil cuando queremos agregar a cada registro información calculada a partir de su grupo, por ejemplo, el promedio de ventas de su categoría.

Lo que devuelve `groupby` con una agregación tiene la columna de agrupación como índice, no como columna. Eso importa cuando después se quiere cruzar con otra tabla: con `reset_index()` vuelve a ser una columna común.

```python
df.groupby("categoria")["ventas"].agg("mean")
```

```python
df["promedio_categoria"] = df.groupby("categoria")["ventas"].transform("mean")
```

**En resumen:** `agg` **reduce** los datos para obtener un resumen por grupo, mientras que `transform` **mantiene** todas las filas y devuelve el resultado alineado con el DataFrame original.

In [ ]:
# Una métrica por grupo
print(ventas.groupby("categoria")["facturacion"].sum().sort_values(ascending=False)) #sort_values al final para ordenar de mayor a menor

In [ ]:
# Tabla agrupada con varias métricas de una vez, con nombres elegidos por quien escribe
resumen = ventas.groupby("categoria").agg(
    ordenes=("id_orden", "count"),
    facturacion=("facturacion", "sum"),
    ticket_promedio=("facturacion", "mean"),
    ticket_mediano=("facturacion", "median"),
)

resumen["participacion"] = resumen["facturacion"] / resumen["facturacion"].sum()

# Visualización incorrecta de resultados (por defecto la que hace la IA)
print("\n", resumen.sort_values("facturacion", ascending=False).round(2))

In [ ]:
# Visualización correcta
display(resumen.sort_values("facturacion", ascending=False).round(2))

**Importante todos las visualizaciones de tablas se deben realizar con display no con print**

Poner el promedio donde correspondía la mediana es el error más común y el más dificil de detectar de los informes de la clase, ya que si no soy conciente que tengo outliers que suben el promedio puedo sobre estimar mi orden normal y sacar proyecciones incorrectas.

In [ ]:
# Crear venta unitaria por categoría

ventas["venta_x_categoria"] = (ventas.groupby("categoria")["id_orden"]
      .transform("count"))

ventas.head()

### Unir fuentes de datos

**`merge`** combina tablas relacionando sus filas a partir de una o más columnas en común. Es similar a realizar un **JOIN en SQL** y permite definir cómo conservar las filas:
- `inner` solamente las que coincidan en ambas tablas (intersección de conjuntos). 
- `left` conserva todas las de la tabla de la izquierda y pone nulos en las columnas de la derecha (conjunto izquierdo + intersección).
- `right` igual que lo anterior pero al revés (conjunto derecho + intersección).
- `outer` conserva todos los registros de ambas tablas rellenando con nulos donde no hay cruces (unión de conjuntos).

**`concat`** une tablas **por filas o por columnas**, manteniendo su estructura. Es útil para apilar datasets con las mismas columnas o para agregar nuevas columnas de distintas fuentes.


In [ ]:
margenes_objetivo = pd.DataFrame({
    "categoria": ["Electronica", "Hogar", "Indumentaria", "Libros", "Deportes"],
    "margen_objetivo": [0.18, 0.32, 0.45, 0.28, 0.38],
})

ventas_con_objetivo = ventas.merge(margenes_objetivo, on="categoria", how="left")

print("Filas antes del cruce:", len(ventas))
print("Filas después del cruce:", len(ventas_con_objetivo))
print("Filas sin objetivo asignado:", ventas_con_objetivo["margen_objetivo"].isna().sum())

El cruce no agregó filas porque la tabla de la derecha tiene un solo registro por categoría. Si tuviera
dos filas para la misma categoría, cada fila de ventas se duplicaría, y la facturación total del informe
saldría inflada sin que nada avise.

De ahí las dos verificaciones que corresponde hacer siempre, y que cuestan dos líneas:

- comparar la cantidad de filas antes y después,
- contar los faltantes en una columna que vino de la tabla derecha, porque cada faltante es una clave que
  no encontró pareja.

Sobre `how`: con `left` se conservan todas las filas de la izquierda y las que no encuentran pareja
quedan con faltantes. Con `inner` se conservan solamente las que encontraron pareja, y las otras
desaparecen sin ningún mensaje. `inner` es el que produce los errores más difíciles de encontrar, porque
el resultado se ve perfecto.

In [ ]:
# Qué pasa cuando la tabla de la derecha tiene claves repetidas
margenes_duplicados = pd.concat([margenes_objetivo, margenes_objetivo.head(2)], ignore_index=True)
display(margenes_duplicados)

# Realizo el mismo cruce que el anterior, pero ahora con claves repetidas en la tabla de la derecha
cruce_inflado = ventas.merge(margenes_duplicados, on="categoria", how="left")

print("Filas después del cruce con claves repetidas:", len(cruce_inflado))
print(f"Facturación real: $U {ventas['facturacion'].sum():,.0f}")
print(f"Facturación inflada: $U {cruce_inflado['facturacion'].sum():,.0f}")

Casi el doble de facturación, con un cruce que no dio ningún error. Ese es el motivo por el que las dos
líneas de verificación no son opcionales.

<h2 style='font-weight:bold; color:#003366;'>2. Retomar el caso donde quedó</h2>

Cada mesa corre la celda de su caso. Reconstruye exactamente lo que quedó de la clase anterior: la carga,
la limpieza y las variables construidas. Vale leerla como repaso, porque es el resumen de la clase 4 en
veinte líneas.

Además cada celda carga las tablas nuevas, que son las que hoy permiten contestar lo que quedó abierto.

In [ ]:
# Caso A: Banco Litoral
TASA_TRANSFERENCIA = 9.5

# El archivo debe tener las columnas creadas de la clase 1 (margen_prestamo, margen_deposito, ingreso_anual)
clientes = (pd.read_csv(ruta_datos("banca_clientes.csv", "casos"), parse_dates=["fecha_alta"])
            .drop_duplicates()
            .copy())

# Tablas nuevas
operaciones = pd.read_csv(ruta_datos("banca_operaciones.csv", "casos"), parse_dates=["fecha"])
costos_canal = pd.read_csv(ruta_datos("banca_costos_canal.csv", "casos"))
sucursales = pd.read_csv(ruta_datos("banca_sucursales.csv", "casos"),
                         parse_dates=["fecha_apertura"])

print(f"clientes: {clientes.shape[0]} filas y {clientes.shape[1]} columnas que son {list(clientes.columns)}")
print(f"operaciones: {operaciones.shape[0]} filas y {operaciones.shape[1]} columnas que son {list(operaciones.columns)}")
print(f"costos_canal: {costos_canal.shape[0]} filas y {costos_canal.shape[1]} columnas que son {list(costos_canal.columns)}")
print(f"sucursales: {sucursales.shape[0]} filas y {sucursales.shape[1]} columnas que son {list(sucursales.columns)}")
print()
display(costos_canal)

In [ ]:
# Caso B: Sierra Norte S.A.
# El archivo debe tener las columnas creadas de la clase 1 (dias_cobro, mes)
facturas = (pd.read_csv(ruta_datos("empresa_facturas.csv", "casos"),
                        parse_dates=["fecha_emision", "fecha_cobro"])
            .drop_duplicates()
            .copy())

# Tablas nuevas
clientes_empresa = pd.read_csv(ruta_datos("empresa_clientes.csv", "casos"),
                               parse_dates=["cliente_desde"])
tipo_cambio = pd.read_csv(ruta_datos("empresa_tipo_cambio.csv", "casos"))
costos_fijos = pd.read_csv(ruta_datos("empresa_costos_fijos.csv", "casos"))

print(f"facturas: {facturas.shape[0]} filas y {facturas.shape[1]} columnas que son {list(facturas.columns)}")
print(f"clientes_empresa: {clientes_empresa.shape[0]} filas y {clientes_empresa.shape[1]} columnas que son {list(clientes_empresa.columns)}")
print(f"tipo_cambio: {tipo_cambio.shape[0]} filas y {tipo_cambio.shape[1]} columnas que son {list(tipo_cambio.columns)}")
print(f"costos_fijos: {costos_fijos.shape[0]} filas y {costos_fijos.shape[1]} columnas que son {list(costos_fijos.columns)}")
print()
display(tipo_cambio.head(3))
display(costos_fijos.head(3))

Una aclaración sobre el caso B antes de empezar: en la clase anterior la conversión de dólares a pesos se
hizo con una constante, 41 pesos por dólar. Hoy está la tabla con la cotización promedio de cada mes, así
que la conversión se puede hacer bien. 

<h2 style='font-weight:bold; color:#003366;'>Etapa 1: el resumen por grupo <span style='font-weight:normal; font-size:70%;'>(12 minutos)</span></h2>

La primera pregunta de cualquier análisis de cartera es dónde está concentrado el resultado. Se contesta
con una tabla agrupada. **Se pide:** Una tabla resumen, con cuatro o cinco métricas por grupo, ordenada por
la que importa, y con una columna de participación sobre el total. Y abajo, en un comentario, la frase de
interpretación.

**Caso A.** Agrupar por segmento. Métricas: cantidad de clientes, ingreso anual total, ingreso anual
promedio, ingreso anual mediano, participación en el ingreso total.

**Caso B.** Agrupar por línea de producto. Métricas: cantidad de facturas, facturación, margen neto,
margen neto sobre facturación, participación en la facturación total. 

**Dos cosas para mirar con atención en el resultado.** La primera: el grupo con más facturación o más
ingreso no suele ser el grupo con mejor promedio, y decidir con uno u otro criterio lleva a decisiones
opuestas. La segunda: dónde el promedio y la mediana se separan mucho, y qué dice eso del grupo.

> **Código de ejemplo**
>
> La forma de pedir varias métricas juntas:
>
> ```python
> resumen = tabla.groupby("columna_de_grupo").agg(
>     nombre_que_quiero=("columna", "función"),
>     otro_nombre=("otra_columna", "otra_función"),
> )
> ```
>
> Las funciones que alcanzan para todo el curso: `"count"`, `"sum"`, `"mean"`, `"median"`, `"min"`,
> `"max"`, `"std"` y `"nunique"`, que cuenta valores distintos.
>
> Ojo con `"count"`: cuenta los valores no faltantes de la columna que se le pase. Si esa columna tiene
> faltantes, el recuento no es la cantidad de filas del grupo. Para la cantidad de filas conviene contar
> sobre la columna del identificador, que nunca está vacía.

In [ ]:
# Caso A
resumen_segmento = clientes.groupby("segmento").agg(
    clientes=(____, ____),
    ingreso_total=(____, "sum"),
    ingreso_promedio=("ingreso_anual", ____),
    ingreso_mediano=("ingreso_anual", ____),
)
resumen_segmento["participacion"] = ____

____(resumen_segmento.sort_values("ingreso_total", ascending=False).round(0))

# Interpretación en un comentario:

In [ ]:
# Caso B
TIPO_CAMBIO_PROVISORIO = 41.0

facturas["factor_moneda"] = np.where(facturas["moneda"] == "USD", TIPO_CAMBIO_PROVISORIO, 1)

cantidad de facturas, facturación, margen neto,
margen neto sobre facturación, participación en la facturación total

resumen_linea = facturas.groupby("linea_producto").agg(
    facturas=(____, ____),
    facturacion=("facturacion", ____),
    margen_neto=(____, "sum"),
    descuento_promedio=("descuento", "mean"),
)

resumen_linea["margen_sobre_facturacion"] = (resumen_linea[____]
                                             / ____["facturacion"])
resumen_linea["participacion"] = (resumen_linea["facturacion"]
                                  / resumen_linea[____].sum())

____(resumen_linea.sort_values(____, ascending=False))

# Interpretación en un comentario:

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b></summary>

La participación es la columna del total dividida por la suma de esa misma columna. Conviene mirar el
resultado con `.round(3)` o formatearlo como porcentaje, porque una participación con doce decimales no
se lee.

Para el caso B, las cinco métricas pedidas se escriben igual que en el caso A. La única que necesita un
paso extra es el margen sobre facturación, que no se puede pedir dentro de `agg` porque es un cociente
entre dos resultados: se calcula después, como columna nueva del resumen.

Y algo que conviene probar en las dos mesas: agregar `.sort_values` por la métrica que importa cambia
por completo la lectura de la tabla. Una tabla ordenada alfabéticamente esconde el hallazgo.

</details>

<h2 style='font-weight:bold; color:#003366;'>Etapa 2: Unir la información de varias fuentes <span style='font-weight:normal; font-size:70%;'></span></h2>

**Caso A.** El ingreso por cliente ya está, pero atender a un cliente cuesta plata y ese costo depende
del canal que use. Un trámite en sucursal cuesta 214 pesos y el mismo trámite en la aplicación cuesta
3,20. La tabla `operaciones` tiene un mes de movimientos y `costos_canal` tiene el costo unitario de cada
canal. El objetivo es llegar a una columna de costo de atención anual por cliente, pegada a la tabla de
clientes.

El camino tiene tres pasos y conviene hacerlos en este orden: 
1. pegarle a cada operación el costo de su canal
2. sumar por cliente
3. pegar ese total a la tabla de clientes. 

Cada paso es un `merge` o un `groupby`, y hay que decidir en cada uno qué pasa con las claves que no tienen pareja.

**Caso B.** La facturación en dólares está convertida con una constante que alguien eligió. La tabla
`tipo_cambio` tiene la cotización promedio de cada mes y la columna `mes` de facturas ya está en el mismo
formato. El objetivo es recalcular la facturación con la cotización correcta y medir la diferencia contra
la versión anterior. Después, pegar la tabla de clientes para tener el sector de cada factura, que es lo
que permite contestar la pregunta de la etapa 3.

**Las dos verificaciones obligatorias en cada cruce.** Filas antes contra filas después, y faltantes en
una columna traída de la derecha.

In [ ]:
# Caso A, paso 1: pegarle a cada operación el costo de su canal
operaciones_con_costo = operaciones.merge(costos_canal, on="canal", how="left")

print("Operaciones antes :", len(operaciones))
print("Operaciones después:", len(operaciones_con_costo))
print("Sin costo asignado :", operaciones_con_costo["costo_operacion_uyu"].isna().sum())

# Paso 2: sumar el costo por cliente y anualizarlo.
# La tabla tiene un mes de movimientos, así que hay que decidir cómo se pasa a anual
# y anotar el supuesto que eso implica.
costo_por_cliente = ____

# Paso 3: pegarlo a la tabla de clientes, y verificar
clientes = clientes.merge(____)

print("\nClientes con costo de atención asignado:",
      clientes["costo_atencion_anual"].notna().sum(), "de", len(clientes))

In [ ]:
# Caso A, la pregunta incómoda del cruce: hay operaciones de clientes que no están
# en la tabla de clientes. Cuántas son, de cuántos clientes, y qué representan.
sin_cliente = operaciones[~operaciones["id_cliente"].isin(clientes["id_cliente"])]

print("Operaciones sin cliente en la tabla:", len(sin_cliente))
print("Clientes distintos involucrados    :", sin_cliente["id_cliente"].nunique())
print(sin_cliente["canal"].value_counts())

# Qué habría pasado con un merge interno en el paso 1 al revés, o sea partiendo de clientes:
# escribir el conteo de filas que sobreviven y decidir si esas operaciones deben contarse
# como costo del banco o no.

In [ ]:
# Caso B, paso 1: la cotización de cada mes
filas_antes = len(facturas)
facturas = facturas.merge(tipo_cambio, on="mes", how="left")

print("Filas antes  :", filas_antes)
print("Filas después:", len(facturas))
print("Facturas sin cotización:", facturas["tipo_cambio_uyu_usd"].isna().sum())

# Paso 2: recalcular con la cotización correcta y comparar contra la versión provisoria
facturas["factor_moneda_real"] = ____
facturas["facturacion_real"] = ____

print(f"\nFacturación con tipo de cambio fijo : $U {facturas['facturacion'].sum():,.0f}")
print(f"Facturación con cotización mensual  : $U {facturas['facturacion_real'].sum():,.0f}")

# Paso 3: pegar la tabla de clientes para tener el sector de cada factura
facturas = facturas.merge(____)
print("\nFacturas sin sector asignado:", facturas["sector"].isna().sum())

<h2 style='font-weight:bold; color:#003366;'>Etapa 3: Conclusión final <span style='font-weight:normal; font-size:70%;'></span></h2>

Con las tablas cruzadas, la conclusión de la etapa 1 se puede reescribir. Esta es la etapa donde
aparece el hallazgo que da vuelta el pedido original.

**Caso A.** El comité quiere cerrar las sucursales del interior. Ahora hay costo de atención por cliente,
así que se puede calcular la rentabilidad neta, que es el ingreso anual menos ese costo. Tres preguntas,
en este orden:

1. Qué segmentos tienen rentabilidad neta negativa, y cuántos clientes son.
2. De dónde viene ese costo: qué canal usan esos clientes comparado con el resto. Para cruzar dos
   variables categóricas y ver la composición porcentual, `pd.crosstab(fila, columna, normalize="index")`
   hace en una línea lo que de otro modo son cuatro.
3. La pregunta que contesta al comité: la rentabilidad por sucursal, y si las del interior son
   efectivamente las peores. Para eso hay que pegar la tabla de sucursales y agrupar por tipo o por
   región.

**Caso B.** El dueño cree que hay que vender más. Ahora está el sector de cada cliente y la fecha de
cobro, así que se puede mirar el otro lado: cuánto tarda en entrar la plata. Tres preguntas:

1. Los días de cobro promedio y mediano por sector, con la cantidad de facturas de cada uno.
2. El saldo sin cobrar por sector, y cuántos días de facturación representa el total sin cobrar. La
   cuenta es el saldo sin cobrar dividido por la facturación diaria promedio del período.
3. Cuántos clientes tienen saldo sin cobrar por encima de su límite de crédito, y cuánto es ese exceso.

In [ ]:
# Caso A
clientes["rentabilidad_neta"] = ____

resumen = clientes.groupby("segmento").agg(
    clientes=("id_cliente", "count"),
    ingreso=("ingreso_anual", "sum"),
    costo_atencion=("costo_atencion_anual", "sum"),
    rentabilidad=("rentabilidad_neta", "sum"),
    rentabilidad_mediana=("rentabilidad_neta", "median"),
)
resumen["clientes_que_pierden"] = clientes.groupby("segmento")["rentabilidad_neta"].apply(
    lambda serie: (serie < 0).mean())

display(resumen.sort_values("rentabilidad").round(2))

In [ ]:
# Caso A, de dónde viene el costo: canal por segmento
operaciones_con_segmento = operaciones.merge(
    clientes[["id_cliente", "segmento"]], on="id_cliente", how="inner")

display(pd.crosstab(operaciones_con_segmento["segmento"],
                  operaciones_con_segmento["canal"],
                  normalize="index").round(3))

# Y la respuesta al comité: rentabilidad por tipo de sucursal y por región
clientes = clientes.merge(sucursales, on="sucursal", how="left")

display("\n", clientes.groupby("tipo_sucursal").agg(
    clientes=("id_cliente", "count"),
    rentabilidad=("rentabilidad_neta", "sum"),
    rentabilidad_por_cliente=("rentabilidad_neta", "mean"),
).round(0))

print("\n", ____)   # lo mismo por región

In [ ]:
# Caso B, pregunta 1: días de cobro por sector
display(facturas.groupby("sector").agg(
    facturas=("id_factura", "count"),
    dias_cobro_promedio=("dias_cobro", "mean"),
    dias_cobro_mediano=("dias_cobro", "median"),
    facturacion=("facturacion_real", "sum"),
).round(1))

# Pregunta 2: el saldo sin cobrar por sector, y cuántos días de facturación representa
sin_cobrar = facturas[facturas["fecha_cobro"].isna()]

display("\n", ____)

dias_del_periodo = (facturas["fecha_emision"].max() - facturas["fecha_emision"].min()).days
facturacion_diaria = facturas["facturacion_real"].sum() / dias_del_periodo
print(f"\nSaldo sin cobrar total: $U {sin_cobrar['facturacion_real'].sum():,.0f}")
print(f"Equivale a {____:.0f} días de facturación")

In [ ]:
# Caso B, pregunta 3: clientes por encima de su límite de crédito
saldo_por_cliente = (sin_cobrar.groupby("id_cliente")["facturacion_real"]
                     .sum()
                     .reset_index()
                     .rename(columns={"facturacion_real": "saldo_abierto"}))

situacion = clientes_empresa.merge(saldo_por_cliente, on="id_cliente", how="left")
situacion["saldo_abierto"] = situacion["saldo_abierto"].fillna(0)
situacion["exceso"] = ____

excedidos = situacion[situacion["exceso"] > 0]
print("Clientes por encima del límite:", len(excedidos), "de", len(situacion))
print(f"Exceso total: $U {excedidos['exceso'].sum():,.0f}")
display(excedidos.nlargest(5, "exceso")[
    ["id_cliente", "sector", "limite_credito_uyu", "saldo_abierto", "exceso"]])

<h2 style='font-weight:bold; color:#003366;'>Etapa 4: concentración y riesgo <span style='font-weight:normal; font-size:70%;'></span></h2>

Una última pregunta antes de armar la recomendación, y es la misma para las dos mesas: cuánto depende el
resultado de unos pocos nombres.

La forma estándar de medirlo es ordenar por la variable de interés, calcular la suma acumulada y
convertirla en porcentaje del total. La cantidad de casos que hacen falta para llegar al 50% es la medida
de concentración que se usa en cualquier informe de cartera.

**Caso A.** Concentración del ingreso por cliente, y concentración del saldo de préstamos dentro de cada
sucursal. La segunda es la que importa desde el punto de vista del riesgo: una sucursal donde un solo
cliente es casi toda la cartera tiene un problema que no se ve en el promedio.

**Caso B.** Concentración de la facturación por cliente, y la tabla de vendedores con el descuento
promedio otorgado y el margen resultante.

In [ ]:
# Caso A
ingreso_ordenado = clientes.sort_values("ingreso_anual", ascending=False)
ingreso_ordenado["acumulado"] = (ingreso_ordenado["ingreso_anual"].cumsum()
                                 / ingreso_ordenado["ingreso_anual"].sum())

print("Clientes necesarios para explicar el 50% del ingreso:",
      (ingreso_ordenado["acumulado"] <= 0.50).sum() + 1, "de", len(clientes))

# Concentración de la cartera de préstamos dentro de cada sucursal.
# El mayor préstamo de la sucursal sobre el total de la sucursal.
concentracion = clientes.groupby("sucursal").agg(
    prestamos=("saldo_prestamo_prom", "sum"),
    mayor_prestamo=("saldo_prestamo_prom", "max"),
    clientes=("id_cliente", "count"),
)
concentracion["peso_del_mayor"] = ____

print("\n", concentracion.sort_values("peso_del_mayor", ascending=False).head(5).round(3))

In [ ]:
# Caso B
facturacion_cliente = (facturas.groupby("id_cliente")["facturacion_real"]
                       .sum()
                       .sort_values(ascending=False))
acumulado = facturacion_cliente.cumsum() / facturacion_cliente.sum()

print("Clientes necesarios para explicar el 50% de la facturación:",
      (acumulado <= 0.50).sum() + 1, "de", facturacion_cliente.size)
print("Peso de los 3 primeros:", f"{acumulado.iloc[2]:.1%}")

# Tabla de vendedores
print("\n", facturas.groupby("vendedor").agg(
    ____
).round(3))

<h2 style='font-weight:bold; color:#003366;'>Etapa 5: la tabla resumen <span style='font-weight:normal; font-size:70%;'></span></h2>

La investigación termina en una tabla que quepa en una diapositiva 
y que sostenga la recomendación: 
entre tres y seis filas, entre tres y cinco columnas,
con nombres de columna que se entiendan sin explicación.

Dos requisitos:

1. Cada fila tiene que ser algo sobre lo que se pueda decidir. Un segmento, una sucursal, un sector, una
   línea de producto. No una fila por cliente.
2. Tiene que haber una columna con el tamaño del grupo, o sea cuántos casos hay atrás. Una tabla de
   promedios sin la cantidad de casos no es defendible.

Debajo de la tabla, la recomendación en tres frases: qué hacer, cuánto vale hacerlo con el número que lo
respalda, y cuál es el riesgo de equivocarse.

In [ ]:
# La tabla final.

**Recomendación de la mesa**

1. Qué hacer:

2. Cuánto vale, con el número que lo respalda:

3. Cuál es el riesgo de equivocarse, y qué dato haría falta para reducirlo:

<h2 style='font-weight:bold; color:#003366;'>Puesta en común y decisión</h2>

**Cada mesa presenta su tabla** 

Algunas preguntas a responder:

1. Ese promedio, cuántos casos tiene atrás, y qué pasa si se saca el caso más grande.
2. Qué decisión de limpieza o qué supuesto sostiene el número, y qué pasa si se cambia.
3. La recomendación, qué costo tiene si el diagnóstico está mal.

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Para seguir en casa</h2>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>

Agrupar por dos columnas a la vez y leer el resultado.

Caso A: rentabilidad neta promedio y cantidad de clientes por segmento y tipo de sucursal.
Caso B: margen sobre facturación y cantidad de facturas por línea de producto y canal de venta.

La pregunta a contestar: aparece alguna combinación que sea claramente peor que las demás, y esa
combinación tiene suficientes casos atrás como para afirmarlo.

In [ ]:
# Tu código acá

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
print(clientes.groupby(["segmento", "tipo_sucursal"]).agg(
    clientes=("id_cliente", "count"),
    rentabilidad_promedio=("rentabilidad_neta", "mean"),
).round(0))

# Caso B
resumen = facturas.groupby(["linea_producto", "canal_venta"]).agg(
    facturas=("id_factura", "count"),
    facturacion=("facturacion_real", "sum"),
    margen=("margen_neto", "sum"),
)
resumen["margen_pct"] = resumen["margen"] / resumen["facturacion"]
print(resumen.round(3))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 **</h3>

La evolución mensual, que es la que contesta si el problema está mejorando o empeorando.

Caso A: no hay serie de tiempo de rentabilidad, así que la pregunta es otra. Agrupar las operaciones por
mes y canal y ver si la mezcla de canales cambia dentro del mes observado.

Caso B: agrupar la facturación y el margen por mes, pegarle la tabla de costos fijos y calcular el
resultado operativo mensual, que es el margen neto menos el costo fijo del mes. Después contestar la
pregunta del dueño con evidencia, comparando los primeros seis meses observados contra los últimos seis:
la facturación, el margen neto y el resultado operativo se mueven en la misma dirección y en la misma
magnitud.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

Caso B: la clave del cruce es la columna de mes, que en `facturas` ya se creó como texto con formato
año-mes y en `costos_fijos` tiene el mismo formato. Conviene verificar que los dos lados tengan la misma
cantidad de valores distintos antes de cruzar, porque un mes sin facturación no va a aparecer en el
`groupby` y va a quedar sin costo fijo asignado, lo que da un resultado operativo equivocado por
omisión.

Para ver si algo crece, ordenar por mes y mirar la serie completa suele alcanzar. Comparar solamente el
primer mes contra el último es lo que hace que dos personas mirando los mismos datos lleguen a
conclusiones opuestas.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
operaciones["semana"] = operaciones["fecha"].dt.isocalendar().week
print(pd.crosstab(operaciones["semana"], operaciones["canal"], normalize="index").round(3))

# Caso B
mensual = facturas.groupby("mes").agg(
    facturacion=("facturacion_real", "sum"),
    margen_neto=("margen_neto_real", "sum"),
    facturas=("id_factura", "count"),
).reset_index()

mensual = mensual.merge(costos_fijos, on="mes", how="left")
print("Meses sin costo fijo asignado:", mensual["costo_fijo_uyu"].isna().sum())

mensual["resultado_operativo"] = mensual["margen_neto"] - mensual["costo_fijo_uyu"]
mensual["margen_pct"] = mensual["margen_neto"] / mensual["facturacion"]
print(mensual[["mes", "facturacion", "margen_neto", "costo_fijo_uyu",
               "resultado_operativo", "margen_pct"]].round(2).to_string(index=False))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 ***</h3>

Reconstruir el resultado principal de la mesa **sin usar la tabla que se agregó hoy**, y comparar las dos
versiones en una sola tabla de dos filas.

Caso A: la rentabilidad por segmento sin el costo de atención, contra la rentabilidad con el costo.
Caso B: el margen por sector con el tipo de cambio fijo en 41, contra el margen con la cotización mensual.

La entrega es la tabla comparativa y una frase sobre qué decisión habría tomado el que pidió el análisis
si le hubiéramos entregado la primera versión.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

No hace falta nada nuevo: son dos `groupby` sobre las mismas filas usando columnas distintas, y después
juntar los dos resultados. Para ponerlos lado a lado, la opción más simple es armar un DataFrame nuevo
con los dos resultados como columnas, aprovechando que los dos tienen el mismo índice de grupos.

El sentido del ejercicio es el que va a aparecer en el proyecto final: casi todos los errores graves de
análisis no son errores de código, son análisis correctos hechos sobre una tabla incompleta.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
comparacion = pd.DataFrame({
    "sin_costo_atencion": clientes.groupby("segmento")["ingreso_anual"].sum(),
    "con_costo_atencion": clientes.groupby("segmento")["rentabilidad_neta"].sum(),
})
comparacion["diferencia_pct"] = (comparacion["con_costo_atencion"]
                                 / comparacion["sin_costo_atencion"] - 1)
print(comparacion.round(3))

# Caso B
facturas["margen_neto_fijo"] = (facturas["facturacion"]
                                - facturas["unidades"] * facturas["costo_unitario"]
                                * facturas["factor_moneda"]
                                - facturas["costo_flete"])
comparacion = pd.DataFrame({
    "tipo_cambio_fijo": facturas.groupby("sector")["margen_neto_fijo"].sum(),
    "cotizacion_mensual": facturas.groupby("sector")["margen_neto_real"].sum(),
})
comparacion["diferencia_pct"] = (comparacion["cotizacion_mensual"]
                                 / comparacion["tipo_cambio_fijo"] - 1)
print(comparacion.round(3))
```

</details>

<h2 style='font-weight:bold; color:#003366;'>Anexo: Conceptos base</h2>

| Qué se quiere | Cómo se escribe |
|---|---|
| Una métrica por grupo | `tabla.groupby("grupo")["monto"].sum()` |
| Varias métricas por grupo | `tabla.groupby("grupo").agg(n=("id", "count"), total=("monto", "sum"))` |
| Agrupar por dos columnas | `tabla.groupby(["grupo", "subgrupo"])["monto"].mean()` |
| Volver el índice a columna | `resultado.reset_index()` |
| Columna calculada sobre el resumen | `resumen["pct"] = resumen["a"] / resumen["a"].sum()` |
| Ordenar el resumen | `resumen.sort_values("total", ascending=False)` |
| Pegar otra tabla | `izquierda.merge(derecha, on="clave", how="left")` |
| Pegar por claves con nombres distintos | `izquierda.merge(derecha, left_on="a", right_on="b", how="left")` |
| Verificar el cruce | comparar `len()` antes y después, y `isna().sum()` de una columna de la derecha |
| Filas de la izquierda sin pareja | `izquierda[~izquierda["clave"].isin(derecha["clave"])]` |
| Composición porcentual cruzada | `pd.crosstab(tabla["a"], tabla["b"], normalize="index")` |
| Valores distintos por grupo | `tabla.groupby("grupo")["cliente"].nunique()` |
| Suma acumulada | `serie.cumsum() / serie.sum()` |
| Apilar dos tablas con las mismas columnas | `pd.concat([tabla1, tabla2], ignore_index=True)` |